# Lab 01: Pandas Index, Series & DataFrame Fundamentals (≈30 minutes)

**Goal:** Build confident, mental models for **Index**, **Series**, and **DataFrame**—and how to select, slice, filter, and reshape using labels vs positions.

> ⏱️ **Timebox**: ~30 minutes total. Each task has an estimate—keep moving if you’re behind.

## Prereqs
- Python 3.9+
- `pandas` (any recent release):
  ```bash
  pip install pandas
  ```
- IDE or notebook (VS Code, Jupyter, Databricks, etc.)

In [ ]:
# Setup (2 min)
import pandas as pd
import numpy as np
pd.__version__

In [ ]:
# Create a small dataset we will reuse
orders_data = {
    "order_id":   [1001, 1002, 1003, 1004, 1005, 1006],
    "customer":   ["Ava", "Ben", "Ava", "Cara", "Ben", "Ava"],
    "city":       ["NYC", "Boston", "NYC", "Austin", "Boston", "NYC"],
    "quantity":   [2, 1, 4, 3, 2, 1],
    "unit_price": [12.5, 9.0, 12.5, 15.0, 9.0, 12.5],
    "order_date": ["2024-01-10", "2024-01-12", "2024-01-12", "2024-02-01", "2024-02-03", "2024-03-15"],
}
df = pd.DataFrame(orders_data)
df["order_date"] = pd.to_datetime(df["order_date"]) 
df.head()

## Task 1 — Series basics (5 min)
A **Series** is a 1-D labeled array (values + index).

In [ ]:
# 1) Create a Series with a custom index and a name.
s = pd.Series([3, 5, 7], index=["x", "y", "z"], name="scores")
s

In [ ]:
# 2) Access items by label and position
print(s.loc["y"])     # 5 (label-based)
print(s.iloc[1])       # 5 (position-based)
s.loc[["z","x"]]      # re-order by labels

In [ ]:
# 3) Inspect metadata and stats
print(s.index, s.dtype, s.name)
display(s.describe())
s.mean()

In [ ]:
# 4) Build a Series from a dict and reindex
prices = pd.Series({"AAPL": 189.3, "MSFT": 418.2, "GOOG": 172.1}, name="price").rename_axis("ticker")
display(prices)
prices2 = prices.reindex(["MSFT", "AAPL", "AMZN", "GOOG"])  # introduces NaN for AMZN
display(prices2)
display(prices2.isna())
prices2.fillna(0.0)

## Task 2 — Index fundamentals (5 min)
The **Index** is the label axis. It’s used for alignment, selection, and joining.

In [ ]:
# 1) Look at the default Index of df
df.index, df.columns

In [ ]:
# 2) Set a meaningful index (primary key) and add a computed column
df = df.set_index("order_id")
df["total"] = df["quantity"] * df["unit_price"]
df.head()

In [ ]:
# 3) Common index ops
print(df.index.name)
df = df.sort_index()
df = df.rename_axis("OrderID")
df.head(3)

In [ ]:
# 4) Reset the index
tmp = df.reset_index()
tmp.head(2)

## Task 3 — DataFrame essentials (8 min)
A **DataFrame** is a 2-D table: columns (named), rows (indexed).

In [ ]:
# 1) Column selection and basic inspection
display(df[["customer","city"]].head(3))
display(df.dtypes)
df.shape

In [ ]:
# 2) Label-based vs position-based row selection
row_by_label = df.loc[1003]
row_by_pos = df.iloc[2]
display(row_by_label)
display(row_by_pos)

In [ ]:
# 3) Slicing differences (inclusive vs exclusive end)
display(df.loc[1002:1005, ["customer","total"]])  # inclusive of 1005
display(df.iloc[1:4, [0, -1]])                      # exclusive of stop

In [ ]:
# 4) Fast scalar access
city = df.at[1004, "city"]
first_val = df.iat[1, 0]
city, first_val

In [ ]:
# 5) Boolean filtering + sorting
nyc_big = df.loc[(df["city"].eq("NYC")) & (df["total"] > 25), ["customer","city","total"]]
nyc_big.sort_values("total", ascending=False)

In [ ]:
# 6) Add/update columns the vectorized way
df["month"] = df["order_date"].dt.to_period("M")
df["discounted"] = np.where(df["total"] >= 40, df["total"]*0.9, df["total"])
df.head()

## Task 4 — MultiIndex mini-exercise (5 min)
A **MultiIndex** lets you index by multiple keys (e.g., `customer` and `city`).

In [ ]:
# 1) Create a MultiIndex
midx = df.reset_index().set_index(["customer","city","OrderID"]).sort_index()
midx.head()

In [ ]:
# 2) Select all orders for a customer in a city (label tuple)
midx.loc[("Ava","NYC")]

In [ ]:
# 3) Select by one level using .xs (cross-section)
ava_all = midx.xs("Ava", level="customer")
nyc_all = midx.xs("NYC", level="city")
display(ava_all.head())
display(nyc_all.head())

In [ ]:
# 4) Partial slice across a range of order IDs for one customer/city
midx.loc[("Ava","NYC")].loc[1002:1006]

## Stretch (optional, 3 min) — Time index
Working with dates is easier if dates are the index.

In [ ]:
t = df.set_index("order_date").sort_index()
t["quantity"].resample("MS").sum()

## Quick self-check (2 min)
Answer without running code (then verify):

1) Which accessor is *inclusive* at the end for slicing: `.loc` or `.iloc`?

2) What happens when you `reindex` with labels not present in the original Series?

3) Which is faster for single-cell gets: `.loc` or `.at`?

4) How do you select all rows for `customer="Ben"` in the MultiIndex DataFrame?

**Answers:**
1) `.loc` is inclusive; `.iloc` is exclusive at the stop.
2) New labels appear with `NaN` (unless you provide `fill_value` or a method).
3) `.at` (label) and `.iat` (position) are optimized for scalars.
4) `midx.xs("Ben", level="customer")` (or `midx.loc["Ben"]` if it’s the outermost level).

## (Optional) Challenge
- Add a `category` column: map `unit_price >= 12` → `"Premium"` else `"Standard"`.
- Compute total revenue by `(customer, category)` using `groupby(["customer","category"])["total"].sum()`.
- Which two orders had the highest `discounted` value? Return `OrderID` and `discounted` only.

In [ ]:
# Challenge (examples)
df["category"] = np.where(df["unit_price"] >= 12, "Premium", "Standard")
revenue_by_cc = df.groupby(["customer","category"])["total"].sum().sort_values(ascending=False)
top2_discounted = df["discounted"].nlargest(2)
df.loc[df["discounted"].nlargest(2).index, ["discounted"]]